# M5.1 · OpenVINO CPU 文本编码器 —— 恢复与公平 A/B

**学习目标：** 验证 OpenVINO BF16 CPU 编码器的恢复（A 部分：本地编码器速度）、真实的端到端架构收益（B 部分：CPU TE 卸载的 GPU 流水线），以及区分架构效应与客户端监控效应的**公平因子实验**（C 部分）。编码器加速比 ≠ 图像吞吐收益——永远不要混淆。

参考讲义：[M5.1 讲义](../course/M05_parallel_heterogeneous/M5_1_OPENVINO_CPU_TE.zh-CN.md) · [English](./06_openvino_and_fair_ab.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
RUN_OWN_GPU_ARMS = True   # <- False in GPU modes: reuse the latest physical run (shown run_id)
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="06")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

def want_own_arms():
    """Whether this session executes its own bounded arms (True unless the
    student opted to reuse the latest physical run to save GPU time)."""
    if session.mode == "REFERENCE" or RUN_OWN_GPU_ARMS:
        return True
    print("RUN_OWN_GPU_ARMS=False: reusing the latest physical campaign run(s) "
          "instead of launching new ones (metered-GPU mode)")
    return False

## 0 · 工件准入 —— PASS 与 PASS_WITH_WARNINGS
OV IR 会对照其转换清单做字节级校验（流式 SHA256、源模型摘要链）。存在未锁定字段时结论保持 `PASS_WITH_WARNINGS`——这一区别要保留，绝不允许四舍五入成 PASS。

In [ ]:
def ov_verify():
    import subprocess, sys
    r = subprocess.run([sys.executable, "-c",
        "import os,sys; sys.path.insert(0,'.');"
        "from lab.m51_contract import verify_artifact;"
        "root=os.environ.get('M51_OV_ARTIFACTS', 'runs/m51-ov-artifacts');"
        "print(verify_artifact(root+'/bf16',"
        "['language_model_bf16.xml','language_model_bf16.bin'],"
        "expected_identity={'storage':'FP16','runtime_precision':'bf16'}))"],
        cwd=str(ROOT), capture_output=True, text=True, timeout=300)
    print(r.stdout or r.stderr)
    return r.returncode

rc = gpu_cell(ov_verify)
if rc is None:
    print("REFERENCE mode: artifact verification runs on the measurement host; "
          "campaign status recorded bf16 = PASS_WITH_WARNINGS (xml digest "
          "computed but not pinned by the historical manifest)")

## A 部分 · 本地编码器：GPU PyTorch vs CPU PyTorch vs OpenVINO BF16
冻结参考比值（仅编码器、同一 CPU）：OV BF16 比 PyTorch CPU 快约 331–480×。**不要盲目断言你的主机能复现**——请核对下面的当前记录。

In [ ]:
import json
enc = ns.read_json("runs/m51-20261105-cleanroom-01/performance/encoder_ab.json")
for arm in ("G_PyTorch_GPU", "C0_PyTorch_CPU", "C1b_OV_BF16"):
    row = enc["arms"][arm]["prompts"]["p1-zh-scene"]
    print(f"{arm:16s} warm median {row['warm_median_s']:8.3f}s  n={row['n_warm']}")
_stamp(HISTORICAL, "frozen encoder A/B (cleanroom campaign, 16-thread cgroup quota)")

def latest_h3():
    cands = sorted((ROOT / "runs").glob("*H3_OV_BF16/H3_RESULTS.json"))
    if not cands:
        return None, None
    return json.loads(cands[-1].read_text()), cands[-1].parent.name

h3, h3_id = gpu_cell(latest_h3) or (None, None)
if h3 is not None:
    session.note_consumed(h3_id, why="current-campaign OV encoder record")
    print("gates:", json.dumps(h3.get("gates", {}), indent=1))
    _stamp(MEASURED_TODAY, f"current-campaign H3_OV_BF16 gates above ({h3.get('mode')})")
    print("release_status:", h3.get("release_status"))
else:
    print("no current-campaign OV encoder record on this host; Part A stays "
          "reference-scoped here")

## B 部分 · 端到端：GPU 原生（A）vs OpenVINO CPU TE（C）
相同的 GPU、DiT、VAE、调度器、分辨率、提示词、种子与客户端策略；每个臂四个真实请求。诚实的指标是**批次墙钟时间 / 每分钟图像数**，服务延迟单独报告。

In [ ]:
def latest_pipeline_ab():
    cands = sorted((ROOT / "runs").glob("*m51pipe*/pipeline_ab.json")) + \
            sorted((ROOT / "runs").glob("m51-*/pipeline/pipeline_ab.json"))
    if not cands:
        return None
    import json as _j
    return _j.loads(cands[-1].read_text())

pab = gpu_cell(latest_pipeline_ab)
if pab is not None:
    for arm, row in pab["arms"].items():
        if isinstance(row, dict) and "batch_wall_s" in json.dumps(row)[:400]:
            print(arm, {k: row.get(k) for k in ("batch_wall_s", "images_per_min",
                                                "gpu_total_s", "cpu_encode_s")
                        if k in row})
    _stamp(MEASURED_TODAY, "pipeline A/C arms from the current campaign")
else:
    frozen = ns.read_json("runs/m51-20261105-cleanroom-01/pipeline/pipeline_ab.json")
    for arm in frozen["arms"]:
        print("frozen", arm, str(frozen["arms"][arm])[:160])
    _stamp(HISTORICAL, "frozen cleanroom pipeline A/B")

## C 部分 · 公平因子实验 —— 架构 × 客户端监控
拿 C-fast 对比 A-original 会夸大架构效应：A-original 还要支付每请求的监控器拆除成本。2×2 因子设计（A/C × original/fast）把两个轴分开。旧研究发现：客户端匹配时架构收益*较小*，而客户端效应更大——把它们当作假设。

In [ ]:
ff = ns.fair_factorial(clean_head=False)
cells = ff["cells"]
print(f"{'cell':12s} {'n':>2s} {'median_s':>9s} {'img/min':>8s}")
for k, v in cells.items():
    print(f"{k:12s} {v['n']:2d} {v['median_s']:9.2f} {v['images_per_min']:8.2f}")
arch = cells["A_original"]["median_s"] / cells["C_original"]["median_s"]
arch_fair = cells["A_fast"]["median_s"] / cells["C_fast"]["median_s"]
client_a = cells["A_original"]["median_s"] / cells["A_fast"]["median_s"]
client_c = cells["C_original"]["median_s"] / cells["C_fast"]["median_s"]
_stamp(HISTORICAL,
       f"fair factorial: architecture effect {arch:.3f}x (original clients) "
       f"vs {arch_fair:.3f}x (matched fast clients); client effect "
       f"A {client_a:.3f}x / C {client_c:.3f}x")
print("never report C_fast vs A_original as a pure architecture effect")

## 正确性范围 —— 什么保持 FAIL，什么留给人工
1. INT8 编码器臂的数值 **FAIL 永远是 FAIL**——本课程任何地方都不会自动提升质量结论。
2. C1b 所有者验收只限定于**历史评估过的部署**；你的新物理运行需要自己的正确性证据。

> **质量判定永远由人完成。** 任何单元都不会自动批准图像质量结论。历史所有者判定**只**适用于其原始评估的部署范围；新的物理运行需要自己的评审（在此之前保持 `PENDING_HUMAN_REVIEW`）。

## 动手练习
1. 用上面的单元计算“仅编码器加速比”与“流水线加速比”——为什么差一个数量级？
2. Part C 中你会给哪个格子加样以收紧架构效应估计？为什么？
3. 在 55 GiB cgroup 上跑 PyTorch CPU 完整臂之前，你会做怎样的 RAM 准入检查？

In [ ]:
summary = {
    "mode": session.mode,
    "notebook": session.notebook,
    "executions": [e.get("run_id") or e.get("kind") for e in session.executions],
    "consumed": [c.get("run_id") for c in session.consumed],
    "note": "append-only session record; physical evidence lives under runs/",
}
session.finish(summary)

## 验收标准

- [ ] 你能指出哪些数字是**今天在这块 GPU 上实测的**、哪些是
      **历史参考值**、哪些是**估计/模拟**。
- [ ] 你产生的每次物理运行都在 `runs/` 下有可见的 `run_id`。
- [ ] 你没有把 smoke/课堂样本冒充为正式协议结果。
- [ ] 你能解释你报告的每项测量的一个局限性。

## 复现与贡献

```bash
python -m lab doctor                                  # 准入证据 -> runs/doctor.json
python -m lab run --experiment H3_OV_BF16 --date <your-tag>  # 完整研究协议
python -m lab report --scope student --kind all       # 只汇总你自己的 runs
```

提交你的 `runs/<run-id>/*.json` 记录（绝不写 `evidence/`），并在 PR 中说明：
假设 → 协议 → 数字 → 局限。欢迎负结果——测得的退化是科学结果，不是失败。